# MagicBox: LFM2.5 Encoder training

Run on a single-host TPU runtime, including Colab v5e-1 and Kaggle v5e-8.
The notebook detects the TPU device count. Set `RUN_MODE = 'smoke'` for
10 full-model updates with a small batch, or `'full'` for the 3-epoch recipe.
Setup clones a pinned commit from the Minifield training GitHub repository.
It downloads the pinned MagicBox dataset and encoder and trains every parameter.

Smoke and full runs have separate output folders. Each saves optimizer checkpoints,
validation reports, and an inference bundle. Set `RESUME_FROM` to continue an
existing run with the same device count and batch settings.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

RUN_MODE = 'smoke'  # 'smoke' or 'full'.
DEVICES = None  # Detect all visible TPUs; set 1 or 8 to require that exact count.
REQUESTS_PER_DEVICE = 1
SMOKE_STEPS = 10
EPOCHS = 3
SESSION_HOURS = 1 if RUN_MODE == 'smoke' else 8
MICROBATCHES = 1 if RUN_MODE == 'smoke' else 4
ROW_CHUNK = 1 if RUN_MODE == 'smoke' else 4
LEARNING_RATE = 2e-5
VALIDATION_RECORDS = 8 if RUN_MODE == 'smoke' else 256
FINAL_RECORDS = 8 if RUN_MODE == 'smoke' else 0

IS_KAGGLE = Path('/kaggle/working').is_dir()
HOST = Path('/kaggle/working') if IS_KAGGLE else Path('/content')
SCRATCH = Path('/kaggle/temp/magicbox') if IS_KAGGLE else HOST / 'magicbox-scratch'
OUTPUT = None  # Defaults to magicbox-{RUN_MODE}-{DEVICES}dev under HOST.
RESUME_FROM = None  # Previous output folder with the same hardware and batch settings.
DATASET = None  # Override with a local completed dataset directory to skip its download.
DATASET_REPO = 'protodotdesign/magicbox-v1'
DATASET_REVISION = 'f074bb549f16ea091fd8ece12e79652b8082871f'

if RUN_MODE not in ('smoke', 'full'):
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")
for name in ('REQUESTS_PER_DEVICE', 'MICROBATCHES', 'ROW_CHUNK', 'SMOKE_STEPS'):
    if type(globals()[name]) is not int or globals()[name] < 1:
        raise ValueError(f'{name} must be a positive integer')
SCRATCH.mkdir(parents=True, exist_ok=True)
CHECKOUT = SCRATCH / 'minifield-training'
SOURCE_REPO_URL = 'https://github.com/Minifield-Labs/minifield-training.git'
SOURCE_REVISION = '658d749528d85315f96b88ff6454bd74941f5907'
VENV = SCRATCH / 'venv'
PYTHON = VENV / 'bin/python'
MODEL = SCRATCH / 'encoder'
CACHE = SCRATCH / 'arrow'

def run_child(*command, cwd=None, env=None):
    print('Starting:', command[0], flush=True)
    child_env = {**os.environ, 'PYTHONUNBUFFERED': '1', **(env or {})}
    with subprocess.Popen(command, cwd=CHECKOUT if cwd is None else cwd, env=child_env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        lines = []
        for line in process.stdout:
            print(line, end='', flush=True)
            lines.append(line)
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)
    return ''.join(lines)


In [ ]:
if not (CHECKOUT / '.git').is_dir():
    if CHECKOUT.exists():
        raise RuntimeError(f'Expected a clean checkout path: {CHECKOUT}')
    run_child('git', 'clone', '--no-checkout', SOURCE_REPO_URL, str(CHECKOUT),
              cwd=SCRATCH)
else:
    if run_child('git', 'status', '--porcelain').strip():
        raise RuntimeError(f'Checkout contains local changes: {CHECKOUT}')
contains_pin = subprocess.run(
    ['git', 'cat-file', '-e', f'{SOURCE_REVISION}^{{commit}}'],
    cwd=CHECKOUT, capture_output=True, check=False).returncode == 0
if not contains_pin:
    run_child('git', 'fetch', '--depth', '1', SOURCE_REPO_URL, SOURCE_REVISION)
run_child('git', 'checkout', '--detach', SOURCE_REVISION)
actual_revision = run_child('git', 'rev-parse', 'HEAD').strip()
if actual_revision != SOURCE_REVISION:
    raise RuntimeError(f'Expected {SOURCE_REVISION}, got {actual_revision}')
if not (CHECKOUT / 'examples/magicbox/train.py').is_file():
    raise RuntimeError('The pinned checkout is missing the MagicBox trainer')
print('Training source:', SOURCE_REPO_URL, actual_revision)


In [ ]:
UV = SCRATCH / 'tooling/bin/uv'
if not UV.is_file():
    run_child(sys.executable, '-m', 'pip', 'install', '--target',
              str(SCRATCH / 'tooling'), 'uv==0.11.30')
environment = {**os.environ, 'UV_PROJECT_ENVIRONMENT': str(VENV),
               'UV_CACHE_DIR': str(SCRATCH / 'uv-cache')}
run_child(str(UV), 'sync', '--locked', '--no-dev', '--all-extras',
          '--python', '3.12', env=environment)
run_child(str(UV), 'pip', 'install', '--python', str(PYTHON), 'jax[tpu]==0.7.2')
hardware_path = SCRATCH / 'hardware.json'
probe = """
import json
from pathlib import Path
import sys
import jax

devices = jax.devices()
if jax.__version__ != '0.7.2':
    raise RuntimeError('Expected pinned JAX 0.7.2')
if jax.process_count() != 1 or len(devices) != jax.local_device_count():
    raise RuntimeError('This notebook requires all TPU devices on one host')
if not devices or any(device.platform != 'tpu' for device in devices):
    raise RuntimeError('Select a TPU runtime')
hardware = {'devices': len(devices), 'kinds': [d.device_kind for d in devices]}
Path(sys.argv[1]).write_text(json.dumps(hardware))
print(hardware, flush=True)
"""
run_child(str(PYTHON), '-c', probe, str(hardware_path))
hardware = json.loads(hardware_path.read_text())
if DEVICES is None:
    DEVICES = hardware['devices']
if type(DEVICES) is not int or DEVICES < 1 or hardware['devices'] != DEVICES:
    raise ValueError(f"Requested {DEVICES} devices; runtime exposes {hardware['devices']}")
REQUESTS = DEVICES * REQUESTS_PER_DEVICE
if OUTPUT is None:
    OUTPUT = HOST / f'magicbox-{RUN_MODE}-{DEVICES}dev'
OUTPUT = Path(OUTPUT)
print('Mode:', RUN_MODE, 'Devices:', DEVICES,
      'Requests per device:', REQUESTS_PER_DEVICE,
      'Logical batch:', REQUESTS * MICROBATCHES, 'Output:', OUTPUT)


## Dataset and encoder

The default downloads the exact uploaded MagicBox v1 snapshot. Set `DATASET` to a
completed local directory to use an attached dataset instead. The loader verifies
every shard and the tokenizer contract and preserves all 5 splits.
Source and schema lengths keep the dataset's limits in both run modes.


In [ ]:
import shutil

if DATASET is None:
    DATASET = SCRATCH / 'dataset'
    download_data = """
import sys
from huggingface_hub import snapshot_download
snapshot_download(sys.argv[1], repo_type='dataset', revision=sys.argv[2],
                  local_dir=sys.argv[3], allow_patterns=[
                      'manifest.json', 'identity.json', 'report.json', 'sources.json',
                      'tokenizer/*', 'train/*.parquet', 'validation/*.parquet',
                      'calibration/*.parquet', 'test/*.parquet', 'ood/*.parquet'])
"""
    run_child(str(PYTHON), '-c', download_data,
              DATASET_REPO, DATASET_REVISION, str(DATASET))
DATASET = Path(DATASET)
if RESUME_FROM is not None and not OUTPUT.exists():
    shutil.copytree(RESUME_FROM, OUTPUT)
manifest = json.loads((DATASET / 'manifest.json').read_text())
assert manifest['format'] == 'minifield.magicbox/1.0' and manifest['complete'] is True
assert manifest['mode'] == 'full'
SOURCE_TOKENS = manifest['tokenizer']['source_limit']
SCHEMA_TOKENS = manifest['tokenizer']['schema_limit']
by_split = {}
for shard in manifest['shards']:
    by_split[shard['split']] = by_split.get(shard['split'], 0) + shard['rows']
print('Dataset:', DATASET, '\nRecords:', by_split)
print('Logical updates per epoch:', (by_split['train'] + REQUESTS * MICROBATCHES - 1) // (REQUESTS * MICROBATCHES))
download_model = """
import sys
from huggingface_hub import snapshot_download
from minifield_training.models.lfm2_5 import encoder
snapshot_download(encoder.SOURCE.model_id, revision=encoder.SOURCE.revision,
                  allow_patterns=['config.json', 'tokenizer.json', 'model.safetensors'],
                  local_dir=sys.argv[1])
"""
run_child(str(PYTHON), '-c', download_model, str(MODEL))


## Startup checks

A tiny CPU model learns examples for all 4 types and verifies exact checkpoint recovery.
The full encoder then commits up to 2 TPU updates, saves a checkpoint, evaluates
8 validation records, and exports an inference bundle. The next cell reloads that
checkpoint and continues toward the selected run's bound.


In [ ]:
run_child(str(PYTHON), '-m', 'examples.magicbox.smoke',
          env={**os.environ, 'JAX_PLATFORMS': 'cpu'})
command = [str(PYTHON), '-m', 'examples.magicbox.train',
           '--dataset', str(DATASET), '--model-dir', str(MODEL),
           '--output', str(OUTPUT), '--cache', str(CACHE),
           '--devices', str(DEVICES), '--platform', 'tpu', '--requests', str(REQUESTS),
           '--microbatches', str(MICROBATCHES), '--row-chunk', str(ROW_CHUNK),
           '--source-tokens', str(SOURCE_TOKENS), '--schema-tokens', str(SCHEMA_TOKENS),
           '--epochs', str(EPOCHS), '--learning-rate', str(LEARNING_RATE)]

def committed_updates():
    manifests = sorted((OUTPUT / 'checkpoints').glob('step-*/manifest.json'))
    return json.loads(manifests[-1].read_text())['cursor']['next_batch'] if manifests else 0

preflight_target = min(2, SMOKE_STEPS) if RUN_MODE == 'smoke' else 2
preflight_remaining = preflight_target - committed_updates()
if preflight_remaining > 0:
    run_child(*command, '--max-steps', str(preflight_remaining), '--max-hours', '1',
              '--checkpoint-every', '2', '--validation-records', '8', '--final-records', '8')


## Run the selected mode

`smoke` runs up to `SMOKE_STEPS` total updates, including startup updates, then
stops. It uses 1 request per device, 1 microbatch, and schema chunks of 1 by default.
`full` runs 3 epochs with 4 microbatches and schema chunks of 4. On 8 devices its
logical batch is 32 requests. Both modes train all parameters with BF16 activations,
FP32 master weights and Adam moments, and gradient checkpointing.

Full training validates every 250 updates and evaluates all holdouts at epoch completion.
Rerunning resumes the same mode and topology. Switch `RUN_MODE` to `full` on the
larger instance to start a separate full run from the pretrained encoder.


In [ ]:
if RUN_MODE == 'smoke':
    remaining = SMOKE_STEPS - committed_updates()
    if remaining > 0:
        run_child(*command, '--max-steps', str(remaining),
                  '--max-hours', str(SESSION_HOURS), '--checkpoint-every', str(SMOKE_STEPS),
                  '--validation-records', str(VALIDATION_RECORDS),
                  '--final-records', str(FINAL_RECORDS))
    print('Smoke updates committed:', committed_updates(), 'Target:', SMOKE_STEPS)
else:
    run_child(*command, '--max-hours', str(SESSION_HOURS), '--checkpoint-every', '250',
              '--validation-records', str(VALIDATION_RECORDS), '--final-records', str(FINAL_RECORDS))


## Reload and predict

This loads the exported bundle in a fresh process and runs the complete typed decoder.
Change the request to inspect extraction, choice, boolean probabilities, and ordinal scores.


In [ ]:
request = {'state': 'Ada sent 3 reports.', 'questions': {
    'person': {'type': 'extract', 'instructions': 'Who sent the reports?'},
    'topic': {'type': 'choice', 'instructions': 'Classify the message.',
              'criteria': {'reporting': 'Reports and documents', 'payments': 'Billing and payments'}},
    'sent': {'type': 'noul', 'instructions': 'Were reports sent?'},
    'volume': {'type': 'score', 'instructions': 'How many reports were sent?',
               'criteria': ['None', 'One', 'Several']}}}
request_path = OUTPUT / 'sample-request.json'
request_path.write_text(json.dumps(request))
bundle = sorted(OUTPUT.glob('bundle-*'))[-1]
run_child(str(PYTHON), '-m', 'examples.magicbox.predict', '--bundle', str(bundle),
          '--request', str(request_path))
print('Keep this folder for inference:', bundle)
print('Keep this folder to resume training:', OUTPUT / 'checkpoints')
for path in sorted(OUTPUT.glob('final-*.json')):
    print(path.name, json.loads(path.read_text()))
